In [2]:
import subprocess
import os
from pathlib import Path

In [15]:
print(os.getcwd())

/home/midori/Desktop/E-learning_Tracker/Notebook


In [3]:
class HDFSManager:
    def __init__(self, hdfs_base_path='/user/elearning'):
        self.hdfs_base_path = hdfs_base_path
        self.hdfs_raw_path = f'{hdfs_base_path}/raw'
        self.hdfs_processed_path = f'{hdfs_base_path}/processed'
        self.hdfs_results_path = f'{hdfs_base_path}/results'
        
    def run_hdfs_command(self, command):
        """Execute HDFS command and return output"""
        try:
            result = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                check=True
            )
            return result.stdout
        except subprocess.CalledProcessError as e:
            print(f"Error executing command: {command}")
            print(f"Error: {e.stderr}")
            return None
    
    def create_hdfs_directories(self):
        """Create HDFS directory structure"""
        print("\n" + "="*60)
        print("Creating HDFS Directory Structure")
        print("="*60 + "\n")
        
        directories = [
            self.hdfs_base_path,
            self.hdfs_raw_path,
            self.hdfs_processed_path,
            self.hdfs_results_path,
            f'{self.hdfs_results_path}/analytics',
            f'{self.hdfs_results_path}/reports'
        ]
        
        for directory in directories:
            print(f"Creating: {directory}")
            self.run_hdfs_command(f'hdfs dfs -mkdir -p {directory}')
        
        print("\nHDFS directories created successfully!")
        self.list_hdfs_structure()
    
    def upload_data_to_hdfs(self, local_data_dir='Data/raw'):
        """Upload all CSV files from local directory to HDFS"""
        print("\n" + "="*60)
        print("Uploading Data to HDFS")
        print("="*60 + "\n")
        
        if not os.path.exists(local_data_dir):
            print(f"Local directory not found: {local_data_dir}")
            return
        
        csv_files = list(Path(local_data_dir).glob('*.csv'))
        
        if not csv_files:
            print(f"No CSV files found in {local_data_dir}")
            return
        
        for csv_file in csv_files:
            print(f"Uploading: {csv_file.name}")
            command = f'hdfs dfs -put -f {csv_file} {self.hdfs_raw_path}/'
            self.run_hdfs_command(command)
        
        print(f"\nUploaded {len(csv_files)} files to HDFS")
        self.list_files(self.hdfs_raw_path)
    
    def list_hdfs_structure(self):
        """Display HDFS directory structure"""
        print("\nHDFS Directory Structure:")
        print("-" * 60)
        output = self.run_hdfs_command(f'hdfs dfs -ls -R {self.hdfs_base_path}')
        if output:
            print(output)
    
    def list_files(self, hdfs_path):
        """List files in specific HDFS directory"""
        print(f"\nFiles in {hdfs_path}:")
        print("-" * 60)
        output = self.run_hdfs_command(f'hdfs dfs -ls {hdfs_path}')
        if output:
            print(output)
    
    def cat_file(self, hdfs_path, num_lines=10):
        """Display file content from HDFS"""
        print(f"\nFirst {num_lines} lines of {hdfs_path}:")
        print("-" * 60)
        command = f'hdfs dfs -cat {hdfs_path} | head -{num_lines}'
        output = self.run_hdfs_command(command)
        if output:
            print(output)

In [16]:
def test_hdfs_connection():
    """Test if HDFS is running and accessible"""
    print("\nTesting HDFS Connection...")
    try:
        result = subprocess.run(
            'hdfs dfs -ls /',
            shell=True,
            capture_output=True,
            text=True,
            check=True
        )
        print("HDFS is running and accessible!")
        return True
    except subprocess.CalledProcessError:
        print("HDFS is not running or not accessible")
        print("\nPlease start HDFS with:")
        print("  start-dfs.sh")
        print("  start-yarn.sh")
        return False


In [17]:
test_hdfs_connection()


Testing HDFS Connection...
HDFS is running and accessible!


True

In [6]:
hdfs_manager = HDFSManager()

In [7]:
hdfs_manager.create_hdfs_directories()


Creating HDFS Directory Structure

Creating: /user/elearning
Creating: /user/elearning/raw
Creating: /user/elearning/processed
Creating: /user/elearning/results
Creating: /user/elearning/results/analytics
Creating: /user/elearning/results/reports

HDFS directories created successfully!

HDFS Directory Structure:
------------------------------------------------------------
drwxr-xr-x   - midori supergroup          0 2026-04-13 21:44 /user/elearning/processed
drwxr-xr-x   - midori supergroup          0 2026-04-13 21:44 /user/elearning/raw
drwxr-xr-x   - midori supergroup          0 2026-04-13 21:44 /user/elearning/results
drwxr-xr-x   - midori supergroup          0 2026-04-13 21:44 /user/elearning/results/analytics
drwxr-xr-x   - midori supergroup          0 2026-04-13 21:44 /user/elearning/results/reports



In [12]:
hdfs_manager.upload_data_to_hdfs('../Data/raw')


Uploading Data to HDFS

Uploading: students.csv
Uploading: courses.csv
Uploading: modules.csv
Uploading: enrollments.csv
Uploading: progress.csv
Uploading: assessments.csv

Uploaded 6 files to HDFS

Files in /user/elearning/raw:
------------------------------------------------------------
Found 6 items
-rw-r--r--   1 midori supergroup     667055 2026-04-13 21:46 /user/elearning/raw/assessments.csv
-rw-r--r--   1 midori supergroup       1711 2026-04-13 21:46 /user/elearning/raw/courses.csv
-rw-r--r--   1 midori supergroup      69432 2026-04-13 21:46 /user/elearning/raw/enrollments.csv
-rw-r--r--   1 midori supergroup       9841 2026-04-13 21:46 /user/elearning/raw/modules.csv
-rw-r--r--   1 midori supergroup     665901 2026-04-13 21:46 /user/elearning/raw/progress.csv
-rw-r--r--   1 midori supergroup      47967 2026-04-13 21:46 /user/elearning/raw/students.csv



In [13]:
hdfs_manager.list_files('/user/elearning/raw')


Files in /user/elearning/raw:
------------------------------------------------------------
Found 6 items
-rw-r--r--   1 midori supergroup     667055 2026-04-13 21:46 /user/elearning/raw/assessments.csv
-rw-r--r--   1 midori supergroup       1711 2026-04-13 21:46 /user/elearning/raw/courses.csv
-rw-r--r--   1 midori supergroup      69432 2026-04-13 21:46 /user/elearning/raw/enrollments.csv
-rw-r--r--   1 midori supergroup       9841 2026-04-13 21:46 /user/elearning/raw/modules.csv
-rw-r--r--   1 midori supergroup     665901 2026-04-13 21:46 /user/elearning/raw/progress.csv
-rw-r--r--   1 midori supergroup      47967 2026-04-13 21:46 /user/elearning/raw/students.csv



In [14]:
hdfs_manager.cat_file('/user/elearning/raw/students.csv', num_lines=5)


First 5 lines of /user/elearning/raw/students.csv:
------------------------------------------------------------
student_id,name,email,age,gender,country,enrollment_date,education_level,employment_status
STU00001,John Hughes,bdominguez@example.net,31,Other,Japan,2025-05-15,Master,Employed
STU00002,Bruce Gilbert,michael56@example.org,44,Male,Dominica,2024-09-25,Master,Employed
STU00003,Crystal Burgess,ritteralexis@example.net,20,Other,Egypt,2025-11-24,Master,Self-Employed
STU00004,Cameron White,edwardscarrie@example.org,28,Other,French Guiana,2024-07-12,PhD,Unemployed

